# AROUSAL

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from scipy.spatial import distance

# === CONFIG ===
input_dir = 'features_d1_d2_MODIFIED'  # change to your folder
target_type = 'Arousal'       # or 'Arousal'
k = 6
distance_metrics = ['euclidean', 'manhattan', 'minkowski', 'cosine', 'hamming']
output_csv = f'knn_k{k}_{target_type.lower()}_classwise_accuracy_Modified.csv'

# === RESULTS ===
results = []

# === Mahalanobis kNN ===
def mahalanobis_knn(X, y):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    preds_all = np.zeros_like(y)

    for train_idx, test_idx in skf.split(X, y):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test = X[test_idx]

        try:
            VI = np.linalg.pinv(np.cov(X_train.T))
        except:
            print("Skipping fold due to singular matrix")
            continue

        fold_preds = []
        for x in X_test:
            dists = np.array([distance.mahalanobis(x, xi, VI) for xi in X_train])
            neighbors = np.argsort(dists)[:k]
            neighbor_labels = y_train[neighbors]
            pred = np.bincount(neighbor_labels).argmax()
            fold_preds.append(pred)

        preds_all[test_idx] = fold_preds

    return preds_all

# === LOOP THROUGH FILES ===
for filename in os.listdir(input_dir):
    if not filename.endswith('.csv'):
        continue

    channel = filename.split('_')[0]
    df = pd.read_csv(os.path.join(input_dir, filename))
    df = df.drop(columns=['Subject', 'Game','TotalPSD','TotalWaveletEnergy'])

    # Label encode Valence or Arousal
    #le = LabelEncoder()
    #df[target_type] = le.fit_transform(df[target_type])  # 0 = NV, 1 = PV or 0 = LA, 1 = HA

    # Manual Mapping for Valence or Arousal
    if target_type == 'Valence':
        label_map = {'NV': 0, 'PV': 1}
    elif target_type == 'Arousal':
        label_map = {'LA': 0, 'HA': 1}
    else:
        raise ValueError("target_type must be 'Valence' or 'Arousal'")

    df[target_type] = df[target_type].map(label_map)
    print(f"\n🔖 Manual Mapping for {target_type}: {label_map}")

    X = df.drop(columns=['Valence', 'Arousal']).values
    y = df[target_type].values

    # Scale features
    X = StandardScaler().fit_transform(X)

    # 10-fold CV
    skf = StratifiedKFold(n_splits=28, shuffle=True, random_state=42)

    # === Built-in Distance Metrics ===
    for metric in distance_metrics:
        all_preds = np.zeros_like(y)

        for train_idx, test_idx in skf.split(X, y):
            knn = KNeighborsClassifier(n_neighbors=k, metric=metric, weights='uniform')
            knn.fit(X[train_idx], y[train_idx])
            all_preds[test_idx] = knn.predict(X[test_idx])
        
        report = classification_report(y, all_preds, output_dict=True, zero_division=0)
        # Get counts
        class_0_total = np.sum(y == 0)
        class_1_total = np.sum(y == 1)
        class_0_correct = np.sum((y == 0) & (all_preds == 0))
        class_1_correct = np.sum((y == 1) & (all_preds == 1))
        
        results.append({
            'Channel': channel,
            'Metric': metric,
            'Class_0_Total': class_0_total,
            'Class_0_Correct': class_0_correct,
            'Class_1_Total': class_1_total,
            'Class_1_Correct': class_1_correct
        })

    # === Mahalanobis Distance ===
    maha_preds = mahalanobis_knn(X, y)
    report = classification_report(y, maha_preds, output_dict=True, zero_division=0)
    # After mahalanobis_knn(X, y)


    class_0_total = np.sum(y == 0)
    class_1_total = np.sum(y == 1)
    class_0_correct = np.sum((y == 0) & (maha_preds == 0))
    class_1_correct = np.sum((y == 1) & (maha_preds == 1))
    
    results.append({
        'Channel': channel,
        'Metric': 'mahalanobis',
        'Class_0_Total': class_0_total,
        'Class_0_Correct': class_0_correct,
        'Class_1_Total': class_1_total,
        'Class_1_Correct': class_1_correct
    })

    
# === SAVE RESULTS ===
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)
print(f'\n✅ All results saved to: {output_csv}')



🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

🔖 Manual Mapping for Arousal: {'LA': 0, 'HA': 1}

✅ All results saved to: knn_k4_arousal_classwise_accuracy_Modified.csv


# VALENCE

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from scipy.spatial import distance

# === CONFIG ===
input_dir = 'features_d1_d2_MODIFIED'  # change to your folder
target_type = 'Valence'       # or 'Arousal'
k = 6
distance_metrics = ['euclidean', 'manhattan', 'minkowski', 'cosine', 'hamming']
output_csv = f'knn_k{k}_{target_type.lower()}_classwise_accuracy_Modified.csv'

# === RESULTS ===
results = []

# === Mahalanobis kNN ===
def mahalanobis_knn(X, y):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    preds_all = np.zeros_like(y)

    for train_idx, test_idx in skf.split(X, y):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test = X[test_idx]

        try:
            VI = np.linalg.pinv(np.cov(X_train.T))
        except:
            print("Skipping fold due to singular matrix")
            continue

        fold_preds = []
        for x in X_test:
            dists = np.array([distance.mahalanobis(x, xi, VI) for xi in X_train])
            neighbors = np.argsort(dists)[:k]
            neighbor_labels = y_train[neighbors]
            pred = np.bincount(neighbor_labels).argmax()
            fold_preds.append(pred)

        preds_all[test_idx] = fold_preds

    return preds_all

# === LOOP THROUGH FILES ===
for filename in os.listdir(input_dir):
    if not filename.endswith('.csv'):
        continue

    channel = filename.split('_')[0]
    df = pd.read_csv(os.path.join(input_dir, filename))
    df = df.drop(columns=['Subject', 'Game','TotalPSD','TotalWaveletEnergy'])

    # Label encode Valence or Arousal
    #le = LabelEncoder()
    #df[target_type] = le.fit_transform(df[target_type])  # 0 = NV, 1 = PV or 0 = LA, 1 = HA

    # Manual Mapping for Valence or Arousal
    if target_type == 'Valence':
        label_map = {'NV': 0, 'PV': 1}
    elif target_type == 'Arousal':
        label_map = {'LA': 0, 'HA': 1}
    else:
        raise ValueError("target_type must be 'Valence' or 'Arousal'")

    df[target_type] = df[target_type].map(label_map)
    print(f"\n🔖 Manual Mapping for {target_type}: {label_map}")

    X = df.drop(columns=['Valence', 'Arousal']).values
    y = df[target_type].values

    # Scale features
    X = StandardScaler().fit_transform(X)

    # 10-fold CV
    skf = StratifiedKFold(n_splits=28, shuffle=True, random_state=42)

    # === Built-in Distance Metrics ===
    for metric in distance_metrics:
        all_preds = np.zeros_like(y)

        for train_idx, test_idx in skf.split(X, y):
            knn = KNeighborsClassifier(n_neighbors=k, metric=metric, weights='uniform')
            knn.fit(X[train_idx], y[train_idx])
            all_preds[test_idx] = knn.predict(X[test_idx])
        
        report = classification_report(y, all_preds, output_dict=True, zero_division=0)
        # Get counts
        class_0_total = np.sum(y == 0)
        class_1_total = np.sum(y == 1)
        class_0_correct = np.sum((y == 0) & (all_preds == 0))
        class_1_correct = np.sum((y == 1) & (all_preds == 1))
        
        results.append({
            'Channel': channel,
            'Metric': metric,
            'Class_0_Total': class_0_total,
            'Class_0_Correct': class_0_correct,
            'Class_1_Total': class_1_total,
            'Class_1_Correct': class_1_correct
        })

    # === Mahalanobis Distance ===
    maha_preds = mahalanobis_knn(X, y)
    report = classification_report(y, maha_preds, output_dict=True, zero_division=0)
    # After mahalanobis_knn(X, y)


    class_0_total = np.sum(y == 0)
    class_1_total = np.sum(y == 1)
    class_0_correct = np.sum((y == 0) & (maha_preds == 0))
    class_1_correct = np.sum((y == 1) & (maha_preds == 1))
    
    results.append({
        'Channel': channel,
        'Metric': 'mahalanobis',
        'Class_0_Total': class_0_total,
        'Class_0_Correct': class_0_correct,
        'Class_1_Total': class_1_total,
        'Class_1_Correct': class_1_correct
    })

    
# === SAVE RESULTS ===
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)
print(f'\n✅ All results saved to: {output_csv}')
